# 07. 머신러닝 심화 - 실무전용 문제 모음

> **실무 전용 노트북** - 이론 설명 없이 TODO 문제만 모았습니다. (관련 이론 노트북: 07_머신러닝_심화_교차검증_튜닝.ipynb)
이미 개념은 이해했다는 전제로, 손으로 더 많이 풀어보며 완전히 몸에 익히는 것이 목표입니다. 정답을 먼저 보지 말고 반드시 스스로 코드를 작성해본 뒤 펼쳐서 비교하세요.

이론 노트북에서는 Titanic으로 연습했다면, 여기서는 **유방암 진단(breast_cancer) 데이터**로 교차검증·튜닝·파이프라인을 다시 연습합니다.

## 목차
1. 교차검증 연습
2. 하이퍼파라미터 튜닝 연습
3. Pipeline & ColumnTransformer 연습
4. KNNImputer 연습
5. PR Curve & 임계값 튜닝 연습

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/breast_cancer.csv')
X = df.drop(columns=['target'])
y = df['target']
X.shape

## 1. 교차검증 연습

**문제 1.** `StratifiedKFold(n_splits=5)`로 `LogisticRegression(max_iter=5000)`을 교차검증해 평균 accuracy를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # StratifiedKFold는 각 fold의 클래스 비율을 원본과 비슷하게 유지(분류 문제에서 권장)
scores = cross_val_score(LogisticRegression(max_iter=5000), X, y, cv=skf)  # cv에 KFold 객체를 넘기면 그 분할 방식대로 5번 학습·평가를 반복
print(scores.mean())  # 5개 점수의 평균이 train_test_split 한 번보다 더 신뢰할 수 있는 성능 추정치
```

</details>

**문제 2.** `cross_validate`로 `accuracy`와 `recall` 두 지표를 동시에 5-fold 교차검증하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.model_selection import cross_validate
result = cross_validate(LogisticRegression(max_iter=5000), X, y, cv=skf, scoring=['accuracy', 'recall'])  # cross_val_score와 달리 여러 지표를 리스트로 동시에 지정 가능
print(result['test_accuracy'].mean(), result['test_recall'].mean())
```

</details>

## 2. 하이퍼파라미터 튜닝 연습

**문제 3.** `RandomForestClassifier`에 대해 `n_estimators=[50,100,200]`, `max_depth=[3,5,None]`으로 `GridSearchCV(cv=5)`를 수행하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
grid = GridSearchCV(RandomForestClassifier(random_state=42), {'n_estimators': [50, 100, 200], 'max_depth': [3, 5, None]}, cv=5)  # 3x3=9가지 조합을 5-fold로 전수 탐색(총 45번 학습)
grid.fit(X, y)
print(grid.best_params_, grid.best_score_)  # best_params_: 가장 성능이 좋았던 조합, best_score_: 그때의 교차검증 평균 점수
```

</details>

**문제 4.** `RandomizedSearchCV`로 같은 모델을 `n_iter=5`만큼 무작위 탐색하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.model_selection import RandomizedSearchCV
rand = RandomizedSearchCV(RandomForestClassifier(random_state=42), {'n_estimators': list(range(50, 301, 25)), 'max_depth': [3, 5, 7, None]}, n_iter=5, cv=5, random_state=42)  # n_iter=5: 전체 조합 중 5개만 무작위로 뽑아 시도(GridSearchCV보다 빠름)
rand.fit(X, y)
print(rand.best_params_)
```

</details>

## 3. Pipeline & ColumnTransformer 연습

**문제 5.** `StandardScaler`와 `LogisticRegression(max_iter=5000)`을 묶은 Pipeline을 만들어 train/test로 나눠 학습·평가하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
pipe = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000))])  # 스케일링과 모델을 하나로 묶어 fit/predict를 한 번에 처리
pipe.fit(X_train, y_train)  # train에만 fit되고, test에는 자동으로 transform만 적용되어 데이터 누수를 방지
print(pipe.score(X_test, y_test))
```

</details>

**문제 6.** `make_pipeline(StandardScaler(), RandomForestClassifier(random_state=42))`로 파이프라인을 만들어 accuracy를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.pipeline import make_pipeline
pipe2 = make_pipeline(StandardScaler(), RandomForestClassifier(random_state=42))  # RandomForest는 스케일링에 크게 영향받지 않지만, 파이프라인 구조를 통일해두면 모델 교체가 쉬움
pipe2.fit(X_train, y_train)
print(pipe2.score(X_test, y_test))
```

</details>

## 4. KNNImputer 연습

**문제 7.** `X`의 `mean radius` 컬럼에 결측치 10개를 임의로 만든 뒤, `KNNImputer(n_neighbors=5)`로 채우세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.impute import KNNImputer
X_missing = X.copy()  # 원본 X를 보존하기 위해 복사본에서 결측치를 만듦
X_missing.loc[X_missing.sample(10, random_state=1).index, 'mean radius'] = np.nan  # 실습을 위해 일부러 결측치 10개를 임의로 생성
imputer = KNNImputer(n_neighbors=5)  # 결측치가 없는 다른 컬럼들을 기준으로 비슷한 행(이웃) 5개를 찾아 그 평균으로 채움
filled = imputer.fit_transform(X_missing)
print(pd.DataFrame(filled, columns=X.columns)['mean radius'].isnull().sum())  # 0이 나오면 결측치가 모두 채워졌다는 뜻
```

</details>

## 5. PR Curve & 임계값 튜닝 연습

**문제 8.** `RandomForestClassifier`를 학습시켜 PR curve를 그리고 average_precision_score를 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import precision_recall_curve, average_precision_score
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)
proba = rf.predict_proba(X_test)[:, 1]
precision, recall, _ = precision_recall_curve(y_test, proba)  # 임계값을 바꿔가며 계산한 precision/recall 쌍들을 반환(thresholds는 여기선 쓰지 않아 _로 무시)
plt.plot(recall, precision)
plt.title(f'AP={average_precision_score(y_test, proba):.3f}')  # AP(Average Precision)는 PR curve 아래 면적을 하나의 숫자로 요약한 값
plt.show()
```

</details>

**문제 9.** 임계값을 0.3으로 낮췄을 때와 0.5일 때의 recall을 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import recall_score
pred_05 = (proba > 0.5).astype(int)
pred_03 = (proba > 0.3).astype(int)  # 임계값을 0.3으로 낮추면 더 쉽게 양성으로 판단하게 되어 recall이 올라가는 경향이 있음
print(recall_score(y_test, pred_05), recall_score(y_test, pred_03))
```

</details>